# Sentimiento en Comunicados de Bancos Centrales y Transmisión Narrativa

**¿Cómo se transmiten las comunicaciones oficiales, el tono de las ruedas de prensa y las sorpresas de postura monetaria a las tasas de interés y la inflación?**

La banca central moderna depende intensamente de la comunicación pública:
1. **Extracción de Sentimiento Halcón vs. Paloma (*Hawkish / Dovish*)**: Cuantificación del balance de términos restrictivos frente a expansivos en minutas y comunicados siguiendo Apel-Blix-Grimaldi (2014) y Picault-Renault (2017):
   $$ \text{Tono}_t = \frac{\text{Halcón}_t - \text{Paloma}_t}{\text{Halcón}_t + \text{Paloma}_t + \epsilon} $$
2. **Sorpresas Monetarias Narrativas**: Uso de cambios cualitativos de postura como choques identificados.
3. **Proyecciones Locales de Alta Frecuencia**: Estimación de las respuestas al impulso de las tasas interbancarias y la inflación mediante proyecciones locales de Jordà (2005) con errores estándar robustos HAC de Newey-West (`puremacro.lp.lp_hac`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_cwd = Path.cwd()
sys.path.insert(0, str(_cwd if (_cwd / "_nbstyle.py").exists() else _cwd / "notebooks"))
import _nbstyle
_nbstyle.apply_style()

from puremacro.datasets import load_banxico_stance
from puremacro.lp import lp_hac
from puremacro.narrative.indices import tone

## 1. Extracción de Tono en Declaraciones de Política Monetaria

In [ ]:
corpus = [
    ("2021-06-15", "The Committee decided to maintain the target range for the federal funds rate at 0 to 1/4 percent. Progress on vaccinations has reduced the spread of COVID-19, but inflation has risen, largely reflecting transitory factors.", {}),
    ("2021-11-03", "Inflation is elevated, largely reflecting factors that are expected to be transitory. Supply bottlenecks and price pressures have broadened across sectors.", {}),
    ("2022-03-16", "The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run. In support of these goals, the Committee decided to raise the target range for the federal funds rate.", {}),
    ("2022-06-15", "The Committee is strongly committed to returning inflation to its 2 percent objective. Decided to raise interest rates by 75 basis points to curb persistent inflationary pressures and overheating labor markets.", {}),
    ("2022-09-21", "Recent indicators point to modest growth in spending and production. Price stability is the responsibility of the Federal Reserve and serves as the bedrock of our economy.", {}),
    ("2022-12-14", "The Committee anticipates that ongoing increases in the target range will be appropriate in order to attain a stance of monetary policy that is sufficiently restrictive.", {}),
    ("2023-05-03", "Tighter credit conditions for households and businesses are likely to weigh on economic activity, hiring, and inflation. The extent of these effects remains uncertain.", {}),
    ("2023-12-13", "Inflation has eased over the past year but remains elevated. Economic growth has slowed from its strong pace in the third quarter.", {}),
    ("2024-06-12", "Inflation has eased substantially over the past year, but remains above our 2 percent longer-run goal. Modest further progress toward the Committee's 2 percent inflation objective has occurred.", {}),
]

tone_res = tone(
    corpus,
    country="US",
    language="en",
    method="apel_blix_grimaldi",
    normalize="raw",
)
print("Vista Previa del Índice de Tono Apel-Blix-Grimaldi:")
print(tone_res.series.dropna())

## 2. Panel Macroeconómico y Postura Narrativa Mensual

In [ ]:
df_banxico = load_banxico_stance()

data_dir = Path(_cwd / "course" / "data" if (_cwd / "course" / "data").exists() else _cwd / "notebooks" / "course" / "data")
df_rate = pd.read_csv(data_dir / "IR3TIB01MXM156N.csv")
df_rate["date"] = pd.to_datetime(df_rate.iloc[:, 0])
df_rate = df_rate.set_index("date")
df_rate.index = df_rate.index.to_period("M")
df_rate["rate_3m"] = pd.to_numeric(df_rate.iloc[:, 1], errors="coerce")

df_cpi = pd.read_csv(data_dir / "CPALTT01MXM659N.csv")
df_cpi["date"] = pd.to_datetime(df_cpi.iloc[:, 0])
df_cpi = df_cpi.set_index("date")
df_cpi.index = df_cpi.index.to_period("M")
df_cpi["inflation_yoy"] = pd.to_numeric(df_cpi.iloc[:, 1], errors="coerce")

df_lp = pd.concat([df_banxico["banxico_direction"], df_rate["rate_3m"], df_cpi["inflation_yoy"]], axis=1).dropna()
df_lp["narrative_shock"] = df_lp["banxico_direction"].to_numpy(dtype=float)
df_lp = df_lp.reset_index(drop=True)
print("Encabezado del Panel Mensual Alineado:")
print(df_lp.head())

## 3. Estimación de Proyecciones Locales de Jordà (2005)

In [ ]:
irf_rate = lp_hac(
    df=df_lp,
    y="rate_3m",
    x="narrative_shock",
    horizons=range(0, 19),
    n_lags=2,
    controls=["inflation_yoy"],
    alpha=0.10,
)

irf_cpi = lp_hac(
    df=df_lp,
    y="inflation_yoy",
    x="narrative_shock",
    horizons=range(0, 19),
    n_lags=2,
    controls=["rate_3m"],
    alpha=0.10,
)

print("Respuesta de la Tasa de Interés (LP):")
print(irf_rate.head(8))

## 4. Respuestas al Impulso con Bandas HAC al 90%

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Respuesta de la Tasa de Interés
ax1.plot(irf_rate["h"], irf_rate["beta"], color="#1f77b4", lw=2, label="Respuesta Tasa Interbancaria 3M")
ax1.fill_between(irf_rate["h"], irf_rate["lo"], irf_rate["hi"], color="#1f77b4", alpha=0.2, label="Banda HAC 90%")
ax1.axhline(0, color="black", lw=0.8, linestyle="--")
ax1.set_title("Respuesta de la Tasa ante Choque Narrativo Restrictivo", fontsize=11, fontweight="bold")
ax1.set_xlabel("Horizonte (Meses)")
ax1.set_ylabel("Tasa de Interés (puntos %)")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

# Respuesta de la Inflación
ax2.plot(irf_cpi["h"], irf_cpi["beta"], color="#d62728", lw=2, label="Respuesta de la Inflación")
ax2.fill_between(irf_cpi["h"], irf_cpi["lo"], irf_cpi["hi"], color="#d62728", alpha=0.2, label="Banda HAC 90%")
ax2.axhline(0, color="black", lw=0.8, linestyle="--")
ax2.set_title("Enfriamiento de la Inflación ante Restricción", fontsize=11, fontweight="bold")
ax2.set_xlabel("Horizonte (Meses)")
ax2.set_ylabel("Inflación Anual (puntos %)")
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()